# 8. Histograms of the ratio to mu=1

Repeat notebook 6 with R=r_eta/r_1 and z=R/(1+R), retaining its bin counts,
scan curves, axis controls, distributions and estimator diagnostics. Reuse the
completed run from notebooks 1–3: no regeneration or training is needed.
Results use `reference_ratio_` and figures `08_`, preserving studies 04–07.

### Runtime and persistent run

In Colab, choose **Runtime → Change runtime type → GPU** for notebooks 2–3.
Both PyTorch training and JAX likelihood fits use the GPU when available.
`JAX_BACKEND = "auto"` detects the Colab NVIDIA GPU; choose `"gpu"` to require it,
or `"cpu"` for an explicit CPU run. Setup installs matching JAX/JAXlib and CUDA 12
plugin packages, then checks double-precision JIT computation and gradients on
the selected device before training. GPU initialization failures are reported
instead of silently switching to CPU. Restart the runtime once when switching
from the previous CPU-only setup or after a JAX plugin error.
For a runtime-only repair, keep the same run name: saved samples and trained
networks are reused. A changed physics model requires a new run, as below.
The source checkout lives in the temporary runtime; **datasets, checkpoints,
workspace files, and results live in Google Drive**. Use the same `RUN_NAME`
and `MODE` in every notebook. Set `MODE = "smoke"` for a short workflow check;
production generates five million events each for S, SBI and B, and fifty
million each for NI, NI_up and NI_down.
Use `CONFIG_OVERRIDES` to change a computational budget without editing source,
for example `{"quadrature_per_process": 500_000, "epochs": 150}`. Use the same
overrides in all notebooks, and choose a new `RUN_NAME` when changing them.

This repository may be private. Grant Colab access when opening it from GitHub.
For the runtime download, either save a GitHub fine-grained token with **Contents:
read** access to this repository as the Colab secret `GITHUB_TOKEN`, or enter it
in the hidden prompt. The token is used only in a Python HTTP header and is not
written into files, Git remotes, shell arguments, or notebook output. Alternatively,
upload a repository ZIP to `/content/poodemo_source.zip`; no token is then needed.
An existing `/content/poodemo` checkout is reused. Delete that checkout to fetch
new source after updating the repository; your Drive run is separate.
For the enlarged NI bank and wider ratio classifiers, use the new default
`RUN_NAME = "paper-distinct-s-v4"`. Rerun notebooks 01–03; then rerun 04–09
to refresh their downstream results. The physics amplitudes are unchanged, but
the sample/training configuration is new (schema 7). Start a fresh Colab runtime
or refresh the temporary source checkout and restart it before proceeding.
Use the same new run name in every notebook; previous Drive runs remain intact.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "paper-distinct-s-v4"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}
JAX_BACKEND = os.environ.get("POODEMO_JAX_BACKEND", "auto")  # auto, gpu, or cpu

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    requested_backend = JAX_BACKEND.strip().lower()
    if requested_backend not in ("auto", "gpu", "cpu"):
        raise ValueError("JAX_BACKEND must be auto, gpu, or cpu")
    selected_backend = requested_backend
    if requested_backend == "auto":
        try:
            gpu_probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                       capture_output=True, text=True, timeout=10)
            gpu_present = gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
        except (OSError, subprocess.TimeoutExpired):
            gpu_present = False
        selected_backend = "gpu" if gpu_present else "cpu"
    # CUDA-only selection fails visibly if initialization fails. Keep PyTorch's
    # CUDA visibility and avoid JAX reserving most GPU memory before training.
    jax_platform = "cuda" if selected_backend == "gpu" else "cpu"
    os.environ["JAX_PLATFORMS"] = jax_platform
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    REPO_DIR = Path("/content/poodemo")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import tempfile
        import urllib.error
        import urllib.request
        import zipfile

        uploaded_zip = Path("/content/poodemo_source.zip")
        if uploaded_zip.exists():
            archive_bytes = uploaded_zip.read_bytes()
            archive_kind = "zip"
        else:
            archive_url = "https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/main"

            def fetch_source(token=None):
                headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
                if token:
                    headers["Authorization"] = "Bearer " + token
                request = urllib.request.Request(archive_url, headers=headers)
                with urllib.request.urlopen(request, timeout=180) as response:
                    return response.read()

            try:
                archive_bytes = fetch_source()
            except urllib.error.HTTPError as error:
                if error.code not in (401, 403, 404):
                    raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
                token = None
                try:
                    from google.colab import userdata
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    pass
                if not token:
                    token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
                if not token:
                    raise RuntimeError("Upload /content/poodemo_source.zip or provide read access.")
                try:
                    archive_bytes = fetch_source(token)
                except urllib.error.HTTPError as auth_error:
                    raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
                finally:
                    token = None
            archive_kind = "tar"

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            if archive_kind == "zip":
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    for entry in archive.infolist():
                        safe_destination(entry.filename)
                        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
                    archive.extractall(staging)
            else:
                with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                    for entry in archive.getmembers():
                        safe_destination(entry.name)
                        if not (entry.isfile() or entry.isdir()):
                            raise RuntimeError("Links and special files are not supported in the source archive.")
                    archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The uploaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        from importlib.metadata import version, PackageNotFoundError
        # JAX discovers every installed plugin, even when selecting CPU.
        # Retain already-matched CUDA 12 packages when rerunning GPU setup.
        remove_plugins = []
        for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt"):
            try:
                installed_version = version(package)
            except PackageNotFoundError:
                continue
            if not (selected_backend == "gpu" and package.startswith("jax-cuda12-") and installed_version == "0.5.3"):
                remove_plugins.append(package)
        if remove_plugins:
            subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *remove_plugins])
        gpu_requirements = (["jax==0.5.3", "jaxlib==0.5.3",
                             "jax-cuda12-plugin[with-cuda]==0.5.3", "jax-cuda12-pjrt==0.5.3"]
                            if selected_backend == "gpu" else [])
        # Resolve with the base requirements so PyTorch's CUDA dependencies are
        # considered together with JAX's. Avoid blanket --upgrade changes.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                               str(REPO_DIR / "requirements-colab.txt"), *gpu_requirements])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

        # Check the live notebook kernel before any expensive network training.
        import jax
        import jaxlib
        import jax.numpy as jnp
        if jax.__version__ != "0.5.3" or jaxlib.__version__ != "0.5.3":
            raise RuntimeError("JAX packages are stale in this kernel. Restart the runtime and rerun setup.")
        if selected_backend == "gpu":
            for package in ("jax-cuda12-plugin", "jax-cuda12-pjrt"):
                if version(package) != "0.5.3":
                    raise RuntimeError(f"{package} must match JAX 0.5.3. Restart the runtime and rerun setup.")
        jax.config.update("jax_platforms", jax_platform)
        jax.config.update("jax_enable_x64", True)
        try:
            jax_devices = jax.devices()
            if not jax_devices or any(device.platform != selected_backend for device in jax_devices):
                raise RuntimeError(f"Expected {selected_backend} devices; found {jax_devices}")
            probe_value, probe_gradient = jax.jit(jax.value_and_grad(lambda x: jnp.sum(x*x)))(
                jnp.asarray([1., 2., 3.], dtype=jnp.float64))
            probe_gradient.block_until_ready()
            if (float(probe_value) != 14. or list(probe_gradient) != [2., 4., 6.]
                    or probe_gradient.dtype != jnp.float64
                    or any(device.platform != selected_backend for device in probe_gradient.devices())):
                raise RuntimeError("JAX double-precision value/gradient check failed")
        except Exception as error:
            raise RuntimeError(
                f"JAX {selected_backend} startup failed. Select a Colab GPU runtime if requesting GPU, "
                "then restart the runtime and rerun setup. Use JAX_BACKEND='cpu' only for an intentional CPU run."
            ) from error
        print(f"JAX {jax.__version__} / jaxlib {jaxlib.__version__}: backend={selected_backend}, {jax_devices}; float64 JIT/gradient OK")

    # A new editable install is not activated in an already-running kernel.
    # Expose the inner package directly, including after a failed import cached
    # the outer /content/poodemo checkout as a namespace package.
    import importlib
    source_root = str(REPO_DIR.resolve())
    if source_root in sys.path:
        sys.path.remove(source_root)
    sys.path.insert(0, source_root)
    importlib.invalidate_caches()
    cached_package = sys.modules.get("poodemo")
    cached_file = getattr(cached_package, "__file__", None)
    expected_init = REPO_DIR / "poodemo" / "__init__.py"
    if cached_package is not None and (cached_file is None or Path(cached_file).resolve() != expected_init.resolve()):
        for module_name in list(sys.modules):
            if module_name == "poodemo" or module_name.startswith("poodemo."):
                del sys.modules[module_name]
else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))

### Ratio to the fixed physical reference

At nominal NI nuisance, use the analytical **selected, normalized** densities:
\[
R_\eta(x)=\frac{r(x;\eta)}{r(x;1)}
=\frac{p(x;\eta,0)}{p(x;1,0)}
=\frac{D_\eta(x)/\Lambda_\eta}{D_1(x)/\Lambda_1},
\qquad z_\eta(x)=\frac{R_\eta(x)}{1+R_\eta(x)}.
\]
The S reference cancels. Both normalizations use the same final quadrature;
this observable is analytical, not learned. The denominator is always at
**mu=1 and alpha_NI=0**, independent of the fit, anchor and Asimov truth settings.
The physical component coefficients, selected yields and Gaussian NI constraint
are unchanged. The event count retains the Poisson rate information.

For fixed NI and Asimov truth 1, the continuous ratio retains the full comparison
between eta and 1 at any separation. Uniform finite bins can still lose
information. This pairwise statement does not imply sufficiency for all fitted
mu values or for NI profiling. Compare to **analytical unbinned**; the learned
references retain their separate closure interpretations.

At **eta=1**, R=1 and z=1/2 exactly for every event. The code preserves this
collapse, without jitter, score substitution, adaptive bins, or rescaling.
That frozen histogram measures only the total rate. Its local shape information
vanishes, even though the Asimov test statistic for testing 1 is correctly zero.
Near 1 the distribution contracts, so uniform bins can remain very coarse.

### How the observable changes with eta

These panels show expected event yields after preselection, with both the
observable constructed at \(\eta\) and the physical prediction evaluated at
\(\mu=\eta\), at nominal \(\alpha_{NI}=0\). For each bin, the coherent layer is
\[
\nu_{\mathrm{SBI},i}^{(\eta)}
=(\eta-\sqrt\eta)\nu_{S,i}(\eta)
+\sqrt\eta\,\nu_{SBI,i}(\eta)
+(1-\sqrt\eta)\nu_{B,i}(\eta).
\]
It is stacked above \(\nu_{NI,i}(\eta)\). Combine the three coherent terms
**before stacking**: their individual coefficients can be negative, while
their physical sum is nonnegative. The curves are not normalized to unit area.
NI's total yield stays fixed even though its distribution in \(z_\eta\) changes.

The default anchors include the generating value, the nominal total-rate
turnover, and the other equal-rate solution when those lie in the scan range.
These rate landmarks need not coincide with full-likelihood extrema.
Edit `ETA_VALUES` and `HIST_BINS` below to explore other choices. All panels
use the same bins and axes; `LOG_Y = True` exposes the tails (set it to `False`
for linear axes). After running setup, this cell can run independently
of the likelihood scans: it reuses the cached analytical quadrature without
retraining or refitting. The figure uses ATLAS-style formatting for this toy
model and is saved as PDF and PNG in the run's `plots` directory on Drive.

In [ ]:
OBSERVABLE = 'reference_ratio'
PLOT_PREFIX = '08'
OBSERVABLE_LABEL = 'Physical-reference ratio $z_\\eta=R_\\eta/(1+R_\\eta)$'

import mplhep as hep
from poodemo.pipeline import prepare_quadrature, observable_values
from poodemo.inference import histogram_components

quad = prepare_quadrature(run)
selected_s, selected_sbi, selected_b, _ = quad["nominal"] @ quad["weights"]
rate_kappa = (selected_s + selected_b - selected_sbi) / selected_s
rate_turnover = (rate_kappa / 2)**2 if rate_kappa > 0 else np.nan
other_root = rate_kappa - np.sqrt(run.config["asimov_mu"])
rate_partner = other_root**2 if other_root >= 0 else np.nan
candidate_anchors = [run.config["mu_min"], run.config["asimov_mu"], rate_turnover, rate_partner,
                     (run.config["mu_min"] + run.config["mu_max"]) / 2, run.config["mu_max"]]
ETA_VALUES = []
for anchor in candidate_anchors:
    if (np.isfinite(anchor) and run.config["mu_min"] <= anchor <= run.config["mu_max"]
            and not any(np.isclose(anchor, old) for old in ETA_VALUES)):
        ETA_VALUES.append(float(anchor))
    if len(ETA_VALUES) == 4:
        break
ETA_VALUES = sorted(ETA_VALUES)  # Replace with your own anchors to explore.
HIST_BINS = 128
LOG_Y = True
edges = np.linspace(0., 1., HIST_BINS + 1)
distributions = []
for eta in ETA_VALUES:
    z_eta = observable_values(run, quad, eta, observable=OBSERVABLE)
    # The cached fields already include the process yields (lambda factors).
    nu_s, nu_sbi, nu_b, nu_ni = histogram_components(
        z_eta, quad["nominal"], quad["weights"], edges)
    root_eta = np.sqrt(eta)
    coherent = (eta - root_eta)*nu_s + root_eta*nu_sbi + (1. - root_eta)*nu_b
    tolerance = 1e-12 * max(1., np.max(np.abs(coherent)))
    if np.any(coherent < -tolerance):
        raise ValueError(f"Negative coherent prediction at eta={eta}: {coherent.min()}")
    # Remove only floating-point roundoff after checking positivity.
    distributions.append((eta, np.maximum(coherent, 0.), nu_ni))

with plt.style.context(hep.style.ATLAS), plt.rc_context({"font.size": 16}):
    ncols = min(2, len(distributions))
    nrows = (len(distributions) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4.8*nrows),
                             sharex=True, sharey=True, squeeze=False)
    ymax = max(np.max(coherent + ni) for _, coherent, ni in distributions)
    for ax, (eta, coherent, ni) in zip(axes.flat, distributions):
        hep.histplot([ni, coherent], bins=edges, stack=True, histtype="fill",
                     color=["#B8B8B8", "#56B4E9"], edgecolor="black", linewidth=0.7,
                     label=["Non-interfering background", r"Coherent SBI ($\mu=\eta$)"],
                     yerr=False, ax=ax)
        ax.text(0.04, 0.94, fr"$\eta={eta:g},\quad\alpha_{{NI}}=0$",
                transform=ax.transAxes, va="top", fontsize=16)
        ax.set(xlim=(0, 1),
               xlabel=OBSERVABLE_LABEL, ylabel="Expected events / bin")
        if LOG_Y:
            ax.set_yscale("log")
            ax.set_ylim(ymax*1e-5, ymax*30)
        else:
            ax.set_ylim(0, 1.45*ymax)
        ax.tick_params(labelbottom=True, labelleft=True)
        ax.grid(False)
        ax.legend(loc="upper right", fontsize=12, frameon=False)
    for ax in axes.flat[len(distributions):]:
        ax.set_visible(False)
    fig.suptitle("Gaussian-amplitude toy · after preselection", fontsize=19)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    (ROOT / "plots").mkdir(exist_ok=True)
    for extension in ("pdf", "png"):
        fig.savefig(ROOT / "plots" / f"{PLOT_PREFIX}_{OBSERVABLE}_distributions.{extension}",
                    bbox_inches="tight", dpi=160)
    plt.show()

display(pd.DataFrame([
    {"eta": eta, "coherent_SBI_yield": coherent.sum(), "NI_yield": ni.sum(),
     "total_yield": (coherent + ni).sum()}
    for eta, coherent, ni in distributions
]))

In [ ]:
from poodemo.pipeline import observable_values, prepare_quadrature
quad = prepare_quadrature(run)
collapse_rows = []
for eta in [0.99, 0.999, 1.0, 1.001, 1.01]:
    z = observable_values(run, quad, eta, "reference_ratio")
    occupied = np.count_nonzero(np.histogram(z, np.linspace(0, 1, 21))[0])
    collapse_rows.append(dict(eta=eta, z_min=z.min(), z_max=z.max(), occupied_bins_20=occupied))
collapse = pd.DataFrame(collapse_rows)
display(collapse)
assert np.all(observable_values(run, quad, 1., "reference_ratio") == .5)
collapse.to_csv(ROOT / "results" / "reference_ratio_collapse.csv", index=False)

### Freeze the observable inside each fit

For a test of mu_0, construct z at eta=mu_0 and nominal NI, then freeze event
assignments, edges and Asimov counts while fitting mu and, where enabled, NI.
The denominator density stays at (1,0); it is never replaced by the best fit.
The binning study fixes NI, as in notebook 6. At Asimov truth 1, infinitely fine
ratio bins recover the pairwise unbinned test, but not necessarily local Fisher
information at the tested eta. The two diagnostics answer different questions.

### Refresh the unbinned references from the saved networks

This cell reloads the existing ratio checkpoints and rebuilds the workspaces
and fits. It does **not** train any network. It refreshes three distinct curves:

| Curve | Likelihood model | Asimov event weights |
|---|---|---|
| `analytic unbinned` | Analytical intensity | Analytical intensity |
| `learned unbinned (analytic Asimov)` | Learned intensity | Analytical intensity: external closure check |
| `learned unbinned (model Asimov)` | Learned intensity | The same learned intensity: finite-sample model Asimov |

The construction follows the finite-sample Asimov prescription discussed in
[arXiv:2609.14136](https://arxiv.org/abs/2609.14136). Let $a_i$ be the saved
quadrature weights, so that $\int f(x)\,dx\approx\sum_i a_i f(x_i)$, and let

$$
\omega_i=a_i q_S(x_i),\qquad
Z_j=\sum_i\omega_i r_j^{\mathrm{NN}}(x_i),\qquad
\widehat p_j(x_i)=q_S(x_i)\frac{r_j^{\mathrm{NN}}(x_i)}{Z_j}.
$$

Here $q_S$ is the selected S reference density normalized on these nodes, so
$\sum_i \omega_i=1$. The $\omega_i$ are reference-density weights, distinct
from the integration weights $a_i$. The process ratios are already normalized using these **same
final quadrature points**; nuisance-morphed shapes are also normalized on those
points. Those existing normalizers are retained. The model Asimov instead
changes which intensity supplies the event weights:

$$
w_{A,i}=a_i\, \widehat D(x_i;\mu_A,0).
$$

With valid positive predictions and the expected rate evaluated on the same
weighted points, the learned model's finite Asimov likelihood satisfies

$$
-2\log\frac{L_A(\mu,\alpha_{NI})}{L_A(\mu_A,0)}
=2\sum_i a_i\left[\widehat D_i-\widehat D_{A,i}
-\widehat D_{A,i}\log\frac{\widehat D_i}{\widehat D_{A,i}}\right]
+\alpha_{NI}^2 \ge 0.
$$

Thus the generating point is a minimum even for a finite set of nodes and an
imperfect learned ratio. This does not guarantee a unique minimum, agreement
with the analytical curvature, or agreement in the secondary basin. The
learned-model Asimov is an internal consistency check; the analytical-Asimov
curve remains the external test of learned-model closure.

The analytical histogram data below are unchanged. Their full-dimensional
comparison is **`analytic unbinned`**. The two learned curves show separately
the effect of changing the model and of constructing its own finite Asimov data.

In [ ]:
from poodemo.pipeline import run_unbinned

# Reuse existing trained checkpoints; only rebuild normalization, workspaces, and fits.
unbinned = run_unbinned(run)
display(unbinned["diagnostics"])
display(unbinned["scans"])
print("Unbinned references refreshed from the saved models; no networks were trained.")

In [ ]:
from dataclasses import replace
from poodemo.pipeline import run_binning_study

BIN_COUNTS = list(range(4, 61, 4))
# The resolution study shares all other settings with notebook 4.
binning_run = replace(run, config={**run.config, "bin_counts": BIN_COUNTS})
binning = run_binning_study(binning_run, observable="reference_ratio")
display(binning["scans"])
display(binning["fidelity"])
print("Saved ratio study:", ROOT / "results" / "reference_ratio_binning_scans.csv")

In [ ]:
def plot_scans(frame, title, filename, group_columns=None, *, xlim=None, ylim=None):
    """Plot saved scans with optional (min, max) axis limits.

    Omit xlim/ylim to retain the default view. A None endpoint keeps that
    bound unchanged, for example ylim=(0, None) or xlim=(None, 1.2).
    """
    x_column = next((c for c in ["mu", "mu_test", "mu0"] if c in frame.columns), None)
    y_column = next((c for c in ["q", "t_mu", "test_statistic", "ts"] if c in frame.columns), None)
    if x_column is None or y_column is None:
        raise ValueError(f"Expected mu and q scan columns; found {list(frame.columns)}")
    if group_columns is None:
        group_columns = [c for c in ["method", "model", "systematics", "nuisances", "n_bins", "bins", "observable"] if c in frame.columns]
    groups = frame.groupby(group_columns, dropna=False, sort=False) if group_columns else [("scan", frame)]
    fig, ax = plt.subplots()
    for label, group in groups:
        group = group.sort_values(x_column)
        values = label if isinstance(label, tuple) else (label,)
        parts = []
        for column, value in zip(group_columns, values):
            if pd.isna(value):
                continue
            if column == "systematics":
                parts.append("profiled" if value else "stat only")
            elif column in ("n_bins", "bins"):
                parts.append(f"{int(value)} bins")
            else:
                parts.append(str(value))
        ax.plot(group[x_column], group[y_column], marker=".", ms=3, label=" | ".join(parts))
    ax.axhline(1.0, color="0.45", ls=":", lw=1)
    ax.set(xlabel=r"Tested signal strength $\mu_0$", ylabel=r"$t(\mu_0)$", title=title)
    ax.set_ylim(bottom=0)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(fontsize=8)
    fig.tight_layout()
    output = ROOT / "plots"
    output.mkdir(exist_ok=True)
    fig.savefig(output / filename, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
SCAN_BIN_COUNTS = [4, 12, 24, 36, 60]
scans = binning["scans"]
shown_scans = scans.loc[
    (scans["model"].eq("direct reference_ratio histogram") & scans["n_bins"].isin(SCAN_BIN_COUNTS))
    | scans["model"].eq("analytic unbinned")
    | scans["model"].str.startswith("learned unbinned", na=False)
].copy()
SCAN_XLIM = (0.02, 1.15)  # Set None to show the full scan range.
SCAN_YLIM = (0, 8)  # Set None to show the full vertical range.
scan_title = "Ratio histograms: " + ", ".join(map(str, SCAN_BIN_COUNTS)) + " bins"
plot_scans(shown_scans, scan_title, "08_binning_scans.pdf",
           xlim=SCAN_XLIM, ylim=SCAN_YLIM);

### Finite-separation fidelity versus local information

The likelihood scan tests eta against the fixed Asimov truth 1. Its continuous
nominal ratio is sufficient for that pair. The Fisher table instead evaluates
local information at mu=eta: pairwise sufficiency does not guarantee retaining
that derivative. In particular, the frozen eta=1 observable has zero shape
information and one occupied bin. The extended information still includes rate.

Non-nested uniform bin counts need not improve monotonically. Near eta=1,
finite binning can obscure the continuous-ratio advantage. Compare the scan
with `analytic unbinned`, separately from the learned references.

In [ ]:
PLOT_PREFIX = '08'

fidelity = binning["fidelity"]
display(fidelity)
eta_near_truth = min(fidelity["eta"].unique(), key=lambda e: abs(e - run.config["asimov_mu"]))
at_anchor = fidelity[np.isclose(fidelity["eta"], eta_near_truth)].sort_values("n_bins")
y_columns = [c for c in fidelity if "information_fraction" in c]
if y_columns:
    ax = at_anchor.plot(x="n_bins", y=y_columns, marker="o", logx=True)
    ax.set(xlabel="Number of bins", ylabel="Retained Fisher information fraction",
           title=fr"Information retained at $\eta={eta_near_truth:.3g}$")
    ax.figure.tight_layout()
    (ROOT / "plots").mkdir(exist_ok=True)
    ax.figure.savefig(ROOT / "plots" / f"{PLOT_PREFIX}_binning_diagnostics.pdf", bbox_inches="tight")
    plt.show()

### Estimator and local uncertainty versus the frozen observable

Now hold the generating experiment fixed at \((\mu_*,\alpha_{NI})=(\mu_*,0)\)
and vary only the observable anchor \(\eta\). Each row refills the
**same Asimov expectation**, freezes that histogram, and fits physical
\(\mu\), with NI either fixed or profiled. For an identified, correctly
modeled Asimov experiment, \(\hat\mu(\eta)\) should remain at the
generating value. This is not a study of finite-sample fluctuations.

The width is evaluated from the local Fisher information at the fixed
truth, using the extended likelihood and the NI auxiliary constraint:
\[
\sigma_\mu^{\rm fixed}(\eta)=1/\sqrt{I_{\mu\mu}^{(\eta)}},\qquad
\sigma_\mu^{\rm profiled}(\eta)
=\sqrt{[(\mathbf I^{(\eta)})^{-1}]_{\mu\mu}}.
\]
This is the local Asimov curvature of **each frozen-observable fit**.
It is neither the curvature of the scan that changes \(\eta=\mu_0\)
at every point nor a global confidence interval. A second likelihood
minimum can produce additional accepted parameter regions that a local
width cannot describe.

The full unbinned reference does not depend on \(\eta\). For the binned
comparison, profiled widths use nuisance morphing after binning, as in
the spline notebook; its difference from integrating the unbinned morphing is a
separate model effect. Inspect anomalous fitted values for optimizer
failure or multiple equally good minima before interpreting them as
compression bias.

Keep the same estimator resolution as notebook 6: the finest bin count in
the saved run configuration. The BIN_COUNTS override only affects the coarsening
study. The truth remains fixed while eta changes. At eta=1 this is a rate-only
measurement; multiple mu values may have equal rates, so a different fitted
branch need not indicate a failed fit. Local widths do not describe that global
ambiguity. Invalid widths are reported rather than floored.

In [ ]:
from poodemo.diagnostics import run_estimator_study

print("Estimator-study bin count:", max(run.config["bin_counts"]))
estimator_study = run_estimator_study(run, observable="reference_ratio")
display(estimator_study)
invalid_rows = estimator_study[~estimator_study["fit_valid"] | ~estimator_study["width_valid"]]
if len(invalid_rows):
    print("Invalid fits or local widths (omitted from the corresponding curves):")
    display(invalid_rows)
print("Saved estimator table:", ROOT / "results" / "reference_ratio_estimator_eta.csv")

In [ ]:
PLOT_PREFIX = '08'

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, squeeze=False)
truth_mu = float(estimator_study["truth"].iloc[0])
for row, systematic in enumerate([False, True]):
    subset = estimator_study[estimator_study["systematics"] == systematic]
    group_columns = [c for c in ["model", "n_bins"] if c in subset.columns]
    for label, values in subset.groupby(group_columns, dropna=False, sort=False):
        values = values.sort_values("eta")
        label_values = label if isinstance(label, tuple) else (label,)
        parts = [f"{int(value)} bins" if name == "n_bins" else str(value)
                 for name, value in zip(group_columns, label_values)
                 if pd.notna(value) and (name != "n_bins" or float(value) > 0)]
        legend_label = " | ".join(parts)
        line_style = "--" if "unbinned" in str(label_values[0]).lower() else "-"
        sigma = values["sigma_mu"].replace([np.inf, -np.inf], np.nan).where(values["width_valid"])
        axes[row, 0].plot(values["eta"], sigma, line_style, label=legend_label)
        axes[row, 1].plot(values["eta"], values["mu_hat"].where(values["fit_valid"]), line_style, label=legend_label)
    nuisance_label = "NI profiled" if systematic else "NI fixed at zero"
    axes[row, 0].set(title=nuisance_label, ylabel=r"Local width $\sigma_\mu(\eta)$")
    axes[row, 1].axhline(truth_mu, color="black", ls=":", label="Generating value")
    axes[row, 1].set(title=nuisance_label + ": same Asimov experiment", ylabel=r"$\hat\mu(\eta)$")
    finite_estimates = subset.loc[subset["fit_valid"], "mu_hat"].to_numpy()
    finite_estimates = finite_estimates[np.isfinite(finite_estimates)]
    lower = min(truth_mu, finite_estimates.min()) if len(finite_estimates) else truth_mu
    upper = max(truth_mu, finite_estimates.max()) if len(finite_estimates) else truth_mu
    margin = max(.03 * (run.config["mu_max"] - run.config["mu_min"]), .1 * (upper - lower))
    axes[row, 1].set_ylim(lower - margin, upper + margin)
    for ax in axes[row]:
        ax.set_xlabel(r"Frozen observable anchor $\eta$")
        ax.legend(fontsize=7)
fig.tight_layout()
for extension in ("pdf", "png"):
    fig.savefig(ROOT / "plots" / f"{PLOT_PREFIX}_estimator_eta.{extension}", bbox_inches="tight", dpi=160)
plt.show()

Next: **09_reference_ratio_spline_templates.ipynb**. Earlier outputs remain intact.